# L3 · ACT explicado, con nuestro modelo abierto sobre la mesa

**ACT** (*Action Chunking with Transformers*, Zhao et al. 2023,
[arXiv 2304.13705](https://arxiv.org/abs/2304.13705)) es la política que
entrenamos primero. Este notebook responde tres preguntas:

1. **Por qué** ACT tiene la forma que tiene: qué problemas de la imitación
   simple resuelve cada pieza.
2. **Qué espera** como entrada y **qué devuelve**, con las formas exactas de
   cada tensor.
3. **Qué pasa adentro** de una predicción, capa por capa, con nuestro modelo
   real y una imagen real.

Cómo está organizado:

- **Lo principal (secciones 1–6).** Con esto alcanza para entender ACT y usarlo
  bien.
- **Para profundizar (secciones 7–9).** El VAE de nuestro modelo (spoiler: no
  está haciendo nada), cómo ejecutar los bloques en el robot, y qué perilla de
  la configuración hace qué.

Todo corre en la Mac, en CPU.

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo
import json

import einops
import matplotlib.pyplot as plt
import numpy as np
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from toolkit.paths import CHECKPOINTS_DIR, DATASETS_DIR, REPO_ROOT
from toolkit.policy.load import load_policy

plt.rcParams["figure.dpi"] = 110
torch.manual_seed(0)
np.random.seed(0)

POLICY_DIR = CHECKPOINTS_DIR / "act_cheap_v1" / "pretrained_model"
ROOT = DATASETS_DIR / "local" / "pick_mameluco_act_v1"
CAMERA = "observation.images.cam_right_high"

## 1. El problema: imitar no es tan fácil como parece

### La idea obvia: *behavior cloning*

Tenemos demostraciones: pares (lo que el robot veía, lo que el operador hizo).
La idea más directa es tratarlo como aprendizaje supervisado común:

> red neuronal: observación → acción del próximo instante

Se entrena con el error entre la acción predicha y la del operador, igual que
cualquier regresión. Esto se llama *behavior cloning* (BC). Funciona sorprendente
mal en manipulación fina, por tres razones concretas.

### Problema 1: los errores se acumulan (*compounding error*)

En supervisado común cada predicción es independiente: equivocarse en una
imagen no cambia la siguiente. En un robot, **la acción de ahora cambia lo que
vas a ver después**. Un error chico te lleva a un estado un poco distinto de
los que viste en las demostraciones; ahí la red es un poco peor; el próximo
error es un poco más grande... y en unos segundos estás en una situación que
nunca vio.

Juguete para verlo: cada decisión de la política tiene un error chico al
azar. Comparamos decidir en cada paso contra decidir cada 48 pasos (y seguir
un plan de 48 acciones en el medio).

In [ ]:
steps, sigma, trials = 400, 0.02, 200
rng = np.random.default_rng(0)

def drift(decide_every):
    # Cada decisión introduce un error que queda "pegado" a la trayectoria.
    errors = np.zeros((trials, steps))
    for t in range(0, steps, decide_every):
        errors[:, t:] += rng.normal(0, sigma, size=(trials, 1))
    return np.abs(errors).mean(axis=0)

for every, label in [(1, "decidir en cada paso"), (8, "bloques de 8"), (48, "bloques de 48")]:
    plt.plot(drift(every), label=label)
plt.xlabel("paso"); plt.ylabel("desvío medio respecto de la demo"); plt.legend(); plt.title("juguete: acumulación de errores")
plt.tight_layout()

Con bloques de *k* acciones hay *k* veces menos decisiones en el mismo tiempo,
y por lo tanto menos oportunidades de acumular error. Es una simplificación
(en el robot real el error no es puro azar), pero la intuición es la del paper:
**el *chunking* reduce el horizonte efectivo de la tarea**.

### Problema 2: las demostraciones humanas no son todas iguales

Ante la misma imagen, un operador a veces va por la izquierda y a veces por la
derecha. Para una red entrenada con error cuadrático (o absoluto) la mejor
respuesta es **el promedio**... que puede ser ir derecho contra el obstáculo.

In [ ]:
x = np.linspace(0, 1, 50)
left = -0.5 * np.sin(np.pi * x)
right = 0.5 * np.sin(np.pi * x)
plt.figure(figsize=(6, 3))
plt.plot(x, left, "tab:blue", label="demo: por la izquierda")
plt.plot(x, right, "tab:green", label="demo: por la derecha")
plt.plot(x, (left + right) / 2, "tab:red", lw=3, label="lo que aprende una regresión")
plt.scatter([0.5], [0], s=400, c="k", marker="s", label="obstáculo")
plt.legend(fontsize=8); plt.yticks([]); plt.title("multimodalidad"); plt.tight_layout()

A esto se le dice que la distribución de acciones es **multimodal** (tiene
varias "jorobas"). ACT lo ataca con un **VAE**, que veremos abajo; Diffusion
Policy lo ataca de otra forma, y por eso vamos a compararlas.

### Problema 3: las pausas confunden a una política de un paso

Si el operador se queda quieto un segundo, la misma imagen aparece con acción
"quieto" y después con acción "moverse". Una red que sólo mira el presente no
sabe cuándo arrancar. Un bloque de acciones ya trae "quieto, quieto, quieto,
ahora moverse" dentro de la misma predicción.

### Las respuestas de ACT, pieza por pieza

| Problema | Pieza de ACT |
|---|---|
| Errores que se acumulan | Predecir un **bloque** de acciones (*action chunking*) |
| Saltos entre bloques | **Temporal ensembling**: promediar bloques superpuestos (opcional) |
| Demostraciones con varios estilos | **VAE**: una variable latente que captura el "estilo" |
| Juntar imagen y articulaciones | **Transformer** que mezcla todo como una secuencia de tokens |
| Ver bien con pocos datos | Backbone **ResNet18 preentrenado en ImageNet** |

## 2. ACT en un dibujo, con las formas de nuestro modelo

```
 cámara (3×360×640) ──► ResNet18 ──► mapa 512×12×20 ──► proyección ──► 240 tokens de 192
 estado (26)        ─────────────────────────────────────► proyección ──►   1 token  de 192
 latente z (32)     ─────────────────────────────────────► proyección ──►   1 token  de 192
                                                                            ───────────────
                                                                            242 tokens
                                                                                │
                                                       encoder transformer (4 capas)
                                                                                │
             48 "preguntas" aprendidas ──► decoder transformer (1 capa) ◄───────┘
                                                │
                                         48 vectores de 192
                                                │
                                   capa lineal ─► 48 acciones × 26 articulaciones
```

Durante el **entrenamiento** hay una pieza más, el *encoder del VAE*, que mira
las acciones verdaderas y produce la `z`. A la hora de **usar** el modelo esa
pieza no existe: `z` vale cero. Lo vemos en la sección 7.

Las 48 "preguntas" del decoder son el truco de DETR (un detector de objetos de
Facebook): en vez de generar las acciones una por una, el decoder recibe 48
vectores aprendidos, uno por posición en el futuro, y responde las 48 de una.

## 3. Qué espera ACT y qué devuelve

La configuración del modelo declara sus entradas y salidas. LeRobot las llama
*features*.

In [ ]:
loaded = load_policy(POLICY_DIR, device="cpu")
policy, config = loaded.policy, loaded.policy.config
print("entradas:")
for name, feature in config.input_features.items():
    print(f"  {name:36s} tipo {feature.type.name:6s} forma {feature.shape}")
print("salidas:")
for name, feature in config.output_features.items():
    print(f"  {name:36s} tipo {feature.type.name:6s} forma {feature.shape}")
print("observaciones por paso (n_obs_steps):", config.n_obs_steps)
print("acciones por bloque (chunk_size):", config.chunk_size)

- **Una sola observación** (`n_obs_steps = 1`): ACT no mira el pasado. Todo lo
  que sabe sale de la imagen y el estado de *este* instante.
- **48 acciones** hacia el futuro.

### Cómo se arman los ejemplos de entrenamiento

Un cuadro del dataset tiene *una* acción. Pero para entrenar necesitamos *las
próximas 48*. LeRobot lo resuelve con `delta_timestamps`: le decís qué
instantes relativos querés de cada clave y arma la ventana sola.

In [ ]:
fps = 30
dataset = LeRobotDataset(
    "local/pick_mameluco_act_v1", root=ROOT, video_backend="pyav",
    delta_timestamps={"action": [i / fps for i in range(config.chunk_size)]},
)
sample = dataset[1000]
{k: (tuple(v.shape), str(v.dtype)) if hasattr(v, "shape") else v for k, v in sample.items()}

Aparece una clave que no estaba en el dataset: **`action_is_pad`**. ¿Qué pasa
si pedimos las próximas 48 acciones a 10 cuadros del final del episodio? No
existen. LeRobot repite la última acción para completar y marca esos lugares
como "relleno", para que el entrenamiento no los cuente.

In [ ]:
episode = dataset.meta.episodes[3]
near_end = int(episode["dataset_to_index"]) - 10
padded = dataset[near_end]
print("cuadros de relleno:", int(padded["action_is_pad"].sum()), "de", config.chunk_size)

plt.figure(figsize=(9, 2.8))
j = 10
plt.plot(padded["action"][:, j], marker="o", ms=3)
plt.axvspan(10 - 0.5, config.chunk_size - 0.5, color="0.85", label="relleno (action_is_pad)")
plt.title("bloque pedido a 10 cuadros del final · articulación 10"); plt.xlabel("paso del bloque"); plt.legend()
plt.tight_layout()

## 4. Antes de la red: normalización

Las 26 articulaciones viven en escalas distintas (radianes para los brazos, 0–1
para las manos). Las redes aprenden mucho mejor si todo entra centrado en 0 y
con dispersión parecida. El **preprocesador** resta la media y divide por el
desvío de cada dimensión; el **postprocesador** deshace eso a la salida.

Esto vive **fuera** de la red. Si la usás sin ellos, la red recibe números en
una escala que nunca vio (nos pasó en el robot). El notebook L4 lo profundiza.

In [ ]:
observation = {k: sample[k] for k in ["observation.state", CAMERA]}
batch = loaded.preprocessor(observation)
print("estado crudo      :", np.round(sample["observation.state"][:5].numpy(), 3))
print("estado normalizado:", np.round(batch["observation.state"][0, :5].numpy(), 3))
print("imagen cruda      : rango", float(sample[CAMERA].min()), "a", float(sample[CAMERA].max()))
image = batch[CAMERA]
print("imagen normalizada: media", round(float(image.mean()), 3), "· desvío", round(float(image.std()), 3))

La imagen se normaliza con la media y el desvío de **ImageNet**, no con los de
nuestro dataset: así es como el ResNet18 aprendió a ver, y le mantenemos el
idioma.

## 5. Una predicción, capa por capa

Ahora abrimos el modelo y hacemos a mano lo mismo que hace
`predict_action_chunk`, mirando cada tensor. Al final comprobamos que da
exactamente igual.

### 5.1 La visión: de imagen a mapa de características

In [ ]:
model = policy.model
with torch.no_grad():
    feature_map = model.backbone(image)["feature_map"]
print("mapa de características:", tuple(feature_map.shape), " → (lote, canales, alto, ancho)")

El ResNet18 achica la imagen 32 veces en cada lado (360/32 ≈ 12, 640/32 = 20)
y a cambio describe cada uno de esos 12×20 "parches" con 512 números. Cada
parche ve una región de unos 30×30 píxeles... y bastante contexto alrededor.

¿Qué regiones "activan" más la red? Un mapa de calor del promedio de los 512
canales:

In [ ]:
activation = feature_map[0].mean(dim=0).numpy()
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
axes[0].imshow(sample[CAMERA].permute(1, 2, 0)); axes[0].set_title("lo que ve"); axes[0].axis("off")
axes[1].imshow(sample[CAMERA].permute(1, 2, 0))
axes[1].imshow(activation, cmap="inferno", alpha=0.55, extent=(0, 640, 360, 0), interpolation="bilinear")
axes[1].set_title("activación media del backbone (12×20)"); axes[1].axis("off")
plt.tight_layout()

(Es una visualización gruesa: el promedio de canales no es "atención". Pero
suele marcar bordes y objetos con textura.)

### 5.2 Todo se vuelve tokens de 192

El transformer sólo entiende secuencias de vectores del mismo tamaño
(`dim_model = 192`). Cada pieza se proyecta a ese tamaño:

In [ ]:
with torch.no_grad():
    image_tokens = model.encoder_img_feat_input_proj(feature_map)               # (1, 192, 12, 20)
    image_pos = model.encoder_cam_feat_pos_embed(feature_map).to(image_tokens.dtype)
    image_tokens = einops.rearrange(image_tokens, "b c h w -> (h w) b c")       # (240, 1, 192)
    image_pos = einops.rearrange(image_pos, "b c h w -> (h w) b c")

    latent = torch.zeros(1, config.latent_dim)                                  # z = 0 al predecir
    latent_token = model.encoder_latent_input_proj(latent)
    state_token = model.encoder_robot_state_input_proj(batch["observation.state"])

    tokens = torch.cat([latent_token[None], state_token[None], image_tokens])   # (242, 1, 192)
    positions = torch.cat([model.encoder_1d_feature_pos_embed.weight.unsqueeze(1), image_pos])
print("tokens:", tuple(tokens.shape), "= 1 latente + 1 estado + 240 de imagen")

Las **posiciones** le dicen al transformer de dónde vino cada token. Sin ellas,
para él la imagen sería una bolsa desordenada de 240 parches. Para la imagen
son senoidales en 2D (fijas); para el latente y el estado, aprendidas.

### 5.3 Encoder: todos los tokens se miran entre sí

In [ ]:
with torch.no_grad():
    memory = model.encoder(tokens, pos_embed=positions)
print("salida del encoder:", tuple(memory.shape))

Después de 4 capas de atención, cada token "sabe" algo de los demás: el token
del estado se enteró de dónde está la prenda, los parches de la imagen se
enteraron de la pose de los brazos.

### 5.4 Decoder: 48 preguntas, 48 respuestas

In [ ]:
with torch.no_grad():
    queries = torch.zeros(config.chunk_size, 1, config.dim_model)
    decoded = model.decoder(queries, memory, encoder_pos_embed=positions,
                            decoder_pos_embed=model.decoder_pos_embed.weight.unsqueeze(1))
    normalized_chunk = model.action_head(decoded.transpose(0, 1))
    chunk = loaded.postprocessor(normalized_chunk)
print("salida del decoder:", tuple(decoded.shape), "→ acciones normalizadas", tuple(normalized_chunk.shape))

with torch.no_grad():
    reference = loaded.postprocessor(policy.predict_action_chunk(batch))
print("¿igual a predict_action_chunk?", torch.allclose(chunk, reference, atol=1e-5))

Las "preguntas" entran en cero; lo que las distingue es su **posición
aprendida** (`decoder_pos_embed`): la pregunta 0 aprendió a significar "¿qué
hago ahora?", la 47 "¿qué hago dentro de 1,6 s?".

Y comparemos con lo que hizo el operador:

In [ ]:
names = dataset.meta.features["action"]["names"][0]
fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, joint in zip(axes, ["kRightShoulderPitch", "kRightElbow", "kRightHandIndex"]):
    j = names.index(joint)
    ax.plot(sample["action"][:, j], color="0.6", lw=3, label="operador")
    ax.plot(chunk[0, :, j], label="ACT")
    ax.set_title(joint); ax.set_xlabel("paso del bloque")
axes[0].legend(); plt.tight_layout()

## 6. Cómo aprende

El entrenamiento muestra lotes de ejemplos (cuadro + sus 48 acciones futuras) y
ajusta los pesos para achicar la pérdida:

- **L1**: el error absoluto medio entre las 48 acciones predichas y las del
  operador, **sin contar** los lugares de relleno (`action_is_pad`).
- **KL** × `kl_weight`: un término del VAE que veremos en la sección 7.

Así arma los lotes el entrenamiento de LeRobot: un `DataLoader` de PyTorch
común sobre el dataset con `delta_timestamps`.

In [ ]:
loader = torch.utils.data.DataLoader(dataset, batch_size=8, shuffle=True, generator=torch.Generator().manual_seed(0))
raw_batch = next(iter(loader))
train_batch = loaded.preprocessor(raw_batch)
print({k: tuple(v.shape) for k, v in train_batch.items() if hasattr(v, "shape")})

policy.train()
with torch.no_grad():
    loss, parts = policy.forward(train_batch)
policy.eval()
print(f"pérdida total {loss.item():.4f} = L1 {parts['l1_loss']:.4f} + {config.kl_weight} × KL {parts['kld_loss']:.6f}")

Fijate el KL: prácticamente cero. Eso es raro, y es la puerta a la primera
profundización.

---

# Para profundizar

## 7. El VAE de nuestro modelo no está haciendo nada

**La idea del VAE en ACT.** Durante el entrenamiento, un transformer extra (el
*encoder del VAE*) mira las 48 acciones **verdaderas** y las resume en un
vector `z` de 32 números: "el estilo de esta demostración" (por izquierda, por
derecha, rápido, lento...). El decoder recibe `z` y así no necesita promediar
estilos: el estilo se lo dicen.

El término **KL** obliga a que esos `z` se parezcan a ruido normal estándar, para
que al usar el modelo podamos poner `z = 0` (el estilo "promedio" de la
distribución, no el promedio de las acciones) y obtener un estilo válido.

**La prueba.** Si `z` codificara algo, cambiar `z` debería cambiar el bloque.
Probamos varios `z` al azar sobre la misma observación:

In [ ]:
def predict_with_latent(z):
    with torch.no_grad():
        first = torch.cat([model.encoder_latent_input_proj(z)[None], state_token[None], image_tokens])
        out = model.encoder(first, pos_embed=positions)
        dec = model.decoder(queries, out, encoder_pos_embed=positions,
                            decoder_pos_embed=model.decoder_pos_embed.weight.unsqueeze(1))
        return loaded.postprocessor(model.action_head(dec.transpose(0, 1)))[0]

base = predict_with_latent(torch.zeros(1, config.latent_dim))
for scale in [1, 3, 10]:
    z = torch.randn(1, config.latent_dim) * scale
    print(f"z con desvío {scale:2d}: el bloque cambia en promedio {(predict_with_latent(z) - base).abs().mean():.5f} rad")

Con `z` del tamaño para el que fue entrenado (desvío 1) el bloque cambia una
diezmilésima de radián: nada. Sólo con valores absurdos (desvío 10, que nunca
vio) se mueve algo. **En la práctica el decoder ignora `z`.** Esto se llama
*posterior collapse*: el KL empuja tan fuerte (`kl_weight = 10`) que al
encoder le sale más barato no codificar nada.

¿Es un problema? Depende. Con **un solo operador muy consistente** (vimos en el
notebook 04 que la velocidad varía poco entre episodios) quizás no hay
"estilos" para capturar, y ACT funciona como un transformer determinista. Si
grabamos con varios operadores o estrategias, sí importaría.

Experimento candidato para cuando reentrenemos: bajar `kl_weight` y ver si el
KL deja de ser cero y si cambia el resultado. Una variable por vez.

## 8. Cómo ejecutar los bloques en el robot

Tenemos un bloque de 48 acciones cada vez que preguntamos. Hay tres formas de
usarlo:

1. **Ejecutar las 48 y recién ahí preguntar de nuevo** (`n_action_steps = 48`,
   lo que hace nuestro modelo). Barato: una consulta cada 1,6 s.
2. **Ejecutar sólo las primeras *n* y volver a preguntar** (`n_action_steps = n`).
   Reacciona más rápido; cuesta más consultas.
3. **Temporal ensembling**: preguntar **en cada paso** y promediar todas las
   predicciones que hay para "ahora" (la del bloque de hace 1 paso, la de hace
   2, ...), con más peso a las viejas. Más suave; una consulta cada 33 ms.

Recordá del notebook 05: el primer paso de cada bloque nuevo arranca corrido.
Comparemos las tres estrategias offline sobre un episodio no visto: le damos
siempre la observación real y armamos la secuencia de acciones que cada
estrategia habría mandado (tarda unos 20 s).

In [ ]:
from lerobot.policies.act.modeling_act import ACTTemporalEnsembler

split = json.loads((REPO_ROOT / "learning" / "act" / "split.json").read_text())
plain = LeRobotDataset("local/pick_mameluco_act_v1", root=ROOT, video_backend="pyav")
episode_index = split["heldout_episode_indices"][2]
meta = plain.meta.episodes[episode_index]
start, end = int(meta["dataset_from_index"]), int(meta["dataset_to_index"])

chunks = []
with torch.no_grad():
    for i in range(start, end):
        frame = plain[i]
        obs = loaded.preprocessor({k: frame[k] for k in ["observation.state", CAMERA]})
        chunks.append(policy.predict_action_chunk(obs))      # normalizado, (1, 48, 26)
truth = torch.stack([plain[i]["action"] for i in range(start, end)]).numpy()
T = len(chunks)

def execute_every(n):
    out = []
    for t in range(0, T, n):
        out.extend(loaded.postprocessor(chunks[t])[0, :n].numpy())
    return np.array(out[:T])

ensembler = ACTTemporalEnsembler(temporal_ensemble_coeff=0.01, chunk_size=config.chunk_size)
ensembled = np.array([loaded.postprocessor(ensembler.update(c))[0].numpy() for c in chunks])

strategies = {"bloques de 48": execute_every(48), "bloques de 8": execute_every(8), "temporal ensembling": ensembled}
j = names.index("kRightElbow")
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(truth[:, j], color="0.6", lw=4, label="operador")
for label, seq in strategies.items():
    ax.plot(seq[:, j], lw=1.2, label=label)
ax.set_title(f"episodio no visto {episode_index} · {names[j]}"); ax.set_xlabel("cuadro"); ax.legend()
plt.tight_layout()

for label, seq in strategies.items():
    jerk = np.abs(np.diff(seq, axis=0)).max(axis=1)
    print(f"{label:20s} error medio {np.abs(seq - truth).mean():.4f} rad · salto máximo entre cuadros {jerk.max():.3f} rad")

Lo que se ve en este episodio:

- **Bloques de 48:** el primer bloque (cuadros 0–47) empieza a subir el codo
  antes de tiempo, porque el operador se quedó quieto ~1,5 s al principio y ACT
  "adivinó" que ya arrancaba. Al llegar el bloque siguiente, salto de golpe
  hacia abajo. Es el problema de los **extremos quietos** del notebook 04 en
  acción. Saltos así, en el robot, son tirones.
- **Bloques de 8:** sigue mejor (menor error) pero con un serrucho de saltos
  chicos.
- **Temporal ensembling:** la curva más suave por lejos (el salto máximo baja
  más de 10 veces), con error intermedio.

Ojo con la lectura: es *teacher forcing* (siempre la observación real). En el
robot, la suavidad además cambia lo que la cámara ve después. La decisión
final entre estrategias se toma en lazo cerrado.

## 9. Las perillas de la configuración

Nuestro modelo contra los valores por defecto de LeRobot:

In [ ]:
from lerobot.policies.act.configuration_act import ACTConfig

defaults = ACTConfig()
knobs = {
    "chunk_size": "cuántas acciones predice por consulta",
    "n_action_steps": "cuántas ejecuta antes de volver a preguntar",
    "temporal_ensemble_coeff": "None = apagado; 0.01 = el del paper",
    "dim_model": "tamaño de los tokens del transformer",
    "n_heads": "cabezas de atención",
    "dim_feedforward": "ancho de la capa interna de cada bloque",
    "n_encoder_layers": "capas del encoder",
    "n_decoder_layers": "capas del decoder (1 replica un bug del código original)",
    "use_vae": "usar o no el VAE",
    "latent_dim": "tamaño de z",
    "kl_weight": "cuánto empuja el KL",
    "dropout": "regularización",
    "optimizer_lr": "tasa de aprendizaje del transformer",
    "optimizer_lr_backbone": "tasa de aprendizaje del ResNet",
    "vision_backbone": "red de visión",
}
import pandas as pd
pd.DataFrame({"nuestro": {k: getattr(config, k) for k in knobs},
              "default LeRobot": {k: getattr(defaults, k) for k in knobs},
              "qué hace": knobs})

Nuestro modelo es una versión **liviana**: transformer de 192 en vez de 512 y
capa interna de 768 en vez de 3200, bloques de 48 en vez de 100, y una tasa de
aprendizaje 5 veces mayor. Tiene sentido para pocos datos y para el Jetson,
pero cada cambio es una variable a validar si queremos exprimir el modelo.

¿Dónde están los parámetros?

In [ ]:
parts = {"backbone ResNet18": model.backbone, "encoder del VAE (sólo entrenamiento)": model.vae_encoder,
         "encoder": model.encoder, "decoder": model.decoder}
counts = {name: sum(p.numel() for p in module.parameters()) for name, module in parts.items()}
counts["resto (proyecciones, posiciones, cabeza)"] = sum(p.numel() for p in model.parameters()) - sum(counts.values())
pd.Series(counts).apply(lambda n: f"{n / 1e6:.2f} M")

Más de dos tercios son el ResNet18. El "cerebro" que decide (encoder + decoder)
es chico.

## Referencias

- Zhao et al. 2023, *Learning Fine-Grained Bimanual Manipulation with Low-Cost
  Hardware* (el paper de ACT y ALOHA): [arXiv 2304.13705](https://arxiv.org/abs/2304.13705).
- Código de ACT en LeRobot: `lerobot/policies/act/modeling_act.py` y
  `configuration_act.py` (dentro de `.venv`).
- Por qué `n_decoder_layers = 1`: [issue del repo original](https://github.com/tonyzhaozh/act/issues/25).
- DETR, de donde sale la idea de las "preguntas" del decoder:
  [arXiv 2005.12872](https://arxiv.org/abs/2005.12872).